# PACE Phase 10B — Controlled 10-seed GA versus NSGA-II

**Analysis only** — no manuscript, original supplementary files or reviewer-response edits.

This notebook runs a **matched-predictor, matched-initialization, equal unique-sequence evaluation budget** comparison.

* Full PACE models: the original four-member AMP classifier and six pathogen + one HC50 Extra-Trees predictors, refitted using the verified Phase 8 data archive.
* Algorithms: original scalar GA versus 4-objective NSGA-II (mean predicted log MIC, worst-pathogen predicted log MIC, negative log selectivity, known-sequence similarity).
* Exactly **9,000 unique scored sequences per algorithm per seed**, **10 paired seeds**, shared 240-population size, maximum 45 generations.
* **Two identical evaluation/lead-selection protocols for both methods**: (a) scalar-fitness-first; (b) Pareto-front/crowding-first. Both select from their **entire evaluation archive**, with the same strict hard filters and pairwise 3-mer diversity rule.
* Full-archive 3D hypervolume with a fixed reference, per-seed metrics and paired-bootstrap uncertainty.

**What to do:** Run the notebook cells in order. Upload only `PACE_Phase8_ESM2_results.zip` when prompted. If your Colab runtime already has the refitted Phase 9 variables, the notebook can reuse them after verifying the archive fingerprint. Otherwise it will fit them again. The experiment saves results after **each complete seed**; rerunning the experiment cell in the same session skips seeds with existing summaries.

**Scientific caution:** All MIC/HC50/selectivity numbers are predictions. This is a computational method-comparison experiment, not antimicrobial efficacy or toxicity validation. The fixed-weight GA and NSGA-II are deliberately compared under a shared objective scoring system; their selection operators and optimum search preferences differ by design.


In [ ]:
from pathlib import Path
import hashlib
ROOT=Path('/content')
archive=ROOT/'PACE_Phase8_ESM2_results.zip'
if not archive.exists():
    from google.colab import files
    uploaded=files.upload()
    if archive.name not in uploaded:
        candidates=[p for p in ROOT.glob('PACE_Phase8_ESM2_results*.zip')]
        assert candidates, 'Please upload the original PACE_Phase8_ESM2_results.zip archive'
        candidates[0].rename(archive)
sha=hashlib.sha256(archive.read_bytes()).hexdigest()
assert sha=='9daf9f932a07ec46121e9e65508c11d68874ceae5d771ec8dd7a3961342edeea', f'Wrong Phase8 ZIP sha256: {sha}'
print('Verified original Phase8 results ZIP:',archive,sha)


In [ ]:
# Fit/reuse Phase 9 full PACE predictor models. This step does NOT run any new GA.
import hashlib
from pathlib import Path
_needed=['get_scores','cls_models','reg','hem','cache','random_seq','mutate','crossover','pair_sim','fitness','provenance']
_reuse=all(k in globals() for k in _needed)
if _reuse:
    _reuse=(provenance.get('phase8_sha256')==hashlib.sha256(Path('/content/PACE_Phase8_ESM2_results.zip').read_bytes()).hexdigest())
if _reuse:
    print('Reusing verified same-Phase8 refitted predictor suite already present in runtime')
else:
    print('Refitting original 4-classifier + 7-regressor PACE predictor suite...')
    SETUP_SOURCE='"""PACE Phase 9 — Fully refitted GA ablations from verified Phase 8 source output.\nNo manuscript or original author file edits. Quantitative MIC/HC50 labels and 90D features\ncome from user\'s ESM2 Phase 8 archive, and original FASTA labels from fixed-source GitHub.\n"""\nimport io, csv, json, time, zipfile, hashlib, collections, sys, importlib.util\nfrom pathlib import Path\nimport numpy as np\nfrom sklearn.ensemble import (ExtraTreesClassifier,RandomForestClassifier,\n    HistGradientBoostingClassifier,ExtraTreesRegressor)\nfrom sklearn.svm import SVC\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\nimport requests\nROOT=Path(\'/content\');OUT=ROOT/\'pace_phase9_colab\';OUT.mkdir(exist_ok=True)\nZ=ROOT/\'PACE_Phase8_ESM2_results.zip\'\nassert Z.exists(), f\'Missing {Z}; upload Phase8 ESM2 results ZIP first\'\nwith zipfile.ZipFile(Z) as z:\n    manifest=json.loads(z.read(\'manifest.json\'))\n    db=np.load(io.BytesIO(z.read(\'all_embeddings_and_features.npz\')))\n    seq_ref=np.asarray(db[\'sequences\']);X_ref=np.asarray(db[\'X90\'])\n    predtab=z.read(\'quantitative_predictions.csv\').decode()\n    (OUT/\'features_original_source.py\').write_bytes(z.read(\'pace_original_features.py\'))\nnpz_dict={s:i for i,s in enumerate(seq_ref)}\nsha=hashlib.sha256()\nsource={}\nfor name,info in manifest[\'source_files\'].items():\n    print(\'Downloading\',name,flush=True)\n    r=requests.get(info[\'url\'],timeout=120);r.raise_for_status()\n    assert hashlib.sha256(r.content).hexdigest()==info[\'sha256\'],f\'Hash mismatch {name}\'\n    source[name]=r.content.decode(\'utf-8-sig\')\n(OUT/\'input_source_manifest.json\').write_text(json.dumps(manifest,indent=2))\nspec=importlib.util.spec_from_file_location(\'pace_features\',OUT/\'features_original_source.py\')\nfeatures=importlib.util.module_from_spec(spec);spec.loader.exec_module(features)\nALPHABET_CANON=set(\'ACDEFGHIKLMNPQRSTVWY\')\ndef parse_fasta(txt):\n    seqs=[];buf=[]\n    for ln in txt.splitlines():\n        if ln.startswith(\'>\'):\n            if buf:seqs.append(\'\'.join(buf));buf=[]\n        else:buf.append(ln.strip())\n    if buf:seqs.append(\'\'.join(buf))\n    return seqs\ndef clean_many(raw):\n    r=[]\n    for x in parse_fasta(raw):\n        s=features.clean(x)\n        if s is not None and 5<=len(s)<=60:r.append(s)\n    return r\ntrainPos=clean_many(source[\'AMP_train\']);trainNeg=clean_many(source[\'nonAMP_train\'])\ntestPos=clean_many(source[\'AMP_test\']);testNeg=clean_many(source[\'nonAMP_test\'])\nextPos=clean_many(source[\'external_AMP\']);extNeg=clean_many(source[\'external_nonAMP\'])\ndef dedup(pos,neg):\n    d={}\n    for s in pos:d.setdefault(s,1)\n    for s in neg:d.setdefault(s,0)\n    return d\ntrain=dedup(trainPos,trainNeg);test={s:y for s,y in dedup(testPos,testNeg).items() if s not in train}\nseen=set(train)|set(test)\nexternal={s:y for s,y in dedup(extPos,extNeg).items() if s not in seen}\nassert (len(train),sum(train.values()),len(test),sum(test.values()),len(external),sum(external.values()))==(6271,3124,1568,796,1848,671)\ncl={**train,**test}; ic=np.array([npz_dict[s] for s in cl]);yc=np.array(list(cl.values()))\nprint(\'Fitting original PACE four-member soft-voting classifier on 7839 train+heldout peptides\',flush=True)\ncls_models={\n    \'GB\':HistGradientBoostingClassifier(max_iter=400,learning_rate=.07,random_state=2024),\n    \'ET\':ExtraTreesClassifier(n_estimators=400,random_state=2024,n_jobs=4),\n    \'SVM\':Pipeline([(\'s\',StandardScaler()),(\'m\',SVC(C=4,gamma=\'scale\',probability=True,random_state=2024))]),\n    \'RF\':RandomForestClassifier(n_estimators=400,random_state=2024,n_jobs=4)\n}\nfor name,model in cls_models.items():\n    model.fit(X_ref[ic],yc);print(\'classifier fitted\',name,flush=True)\nqs=[r for r in csv.DictReader(io.StringIO(predtab)) if r[\'representation\']==\'90D_PACE\']\nquant={}\nfor r in qs:quant.setdefault(r[\'endpoint\'],{}).setdefault(r[\'sequence\'],float(r[\'observed_log10\']))\nexp={\'E. faecium\':780,\'S. aureus\':4128,\'K. pneumoniae\':743,\'A. baumannii\':306,\'P. aeruginosa\':2517,\'E. cloacae\':304,\'HC50\':1271}\nassert all(len(quant[x])==n for x,n in exp.items()),{x:len(quant.get(x,{})) for x in exp}\nreg={}\nfor target in list(exp)[:-1]:\n    q=quant[target];ix=np.array([npz_dict[s] for s in q]);y=np.array(list(q.values()))\n    m=ExtraTreesRegressor(n_estimators=300,random_state=2024,n_jobs=4);m.fit(X_ref[ix],y);reg[target]=m\n    print(\'MIC fitted\',target,len(y),flush=True)\nq=quant[\'HC50\'];ix=np.array([npz_dict[s] for s in q]);y=np.array(list(q.values()))\nhem=ExtraTreesRegressor(n_estimators=300,random_state=2024,n_jobs=4);hem.fit(X_ref[ix],y)\nprint(\'HC50 fitted\',len(y),flush=True)\nPOOL=sorted(set(s for s,y in {**train,**test,**external}.items() if y==1)|set().union(*(set(q.keys()) for q in quant.values())))\nK=3; inv=collections.defaultdict(list);sets=[]\nfor i,s in enumerate(POOL):\n    a=set(s[j:j+3] for j in range(len(s)-2));sets.append(a)\n    for k in a:inv[k].append(i)\ndef maximum_similarity(s):\n    a={s[j:j+3] for j in range(len(s)-2)};cnt=collections.Counter()\n    for k in a:\n        for i in inv.get(k,()):cnt[i]+=1\n    best=0.\n    for i,shared in cnt.items():\n        j=shared/(len(a)+len(sets[i])-shared)\n        if j>best:best=j\n    return best\ncache={};counts={\'features\':0}\ndef get_scores(seqs):\n    new=[s for s in dict.fromkeys(seqs) if s not in cache]\n    if new:\n        X=features.featurize_many(new)\n        p=np.mean([m.predict_proba(X)[:,1] for m in cls_models.values()],axis=0)\n        mm=np.column_stack([reg[t].predict(X) for t in reg]);hc=hem.predict(X)\n        for i,s in enumerate(new):\n            sim=maximum_similarity(s);mean=float(mm[i].mean());worst=float(mm[i].max());ti=float(hc[i]-mean);charge=features.net_charge(s)\n            cache[s]={\'p_amp\':float(p[i]),\'mic_mean\':mean,\'mic_worst\':worst,\'hc50\':float(hc[i]),\'ti\':ti,\'sim\':sim,\'charge\':float(charge),\'length\':len(s),\'sequence\':s}\n        counts[\'features\']+=len(new)\n    return [cache[s] for s in seqs]\nprovenance={\'source\':\'Analysis-only independent refit of PACE 90D models from Phase8 curated labels and pinned FASTA sources, reconstructed from original published code\',\n  \'quantitative_labels_source\':\'Phase8 quantitative_predictions 90D_PACE observed_log10 (uniquely deduplicated by sequence and endpoint)\',\n  \'classifier_training_n\':len(cl),\'classifier_names\':list(cls_models),\'classifier_trees\':400,\'regressor_trees\':300,\n  \'novelty_known_peptides\':len(POOL),\'ga_seeds\':[7,29],\'original_pace_source\':\'PACE_reproducibility_code.zip\',\n  \'original_pace_code_zip_sha256\':\'8a72718087b551b968ed0c85652ae40dda029ae61a5dc7c7cbff87184eceace7\',\n  \'limits\':\'Newly fitted models may not bitwise reproduce prior author weights; same descriptor code and model settings\',\n  \'phase8_sha256\':hashlib.sha256(Z.read_bytes()).hexdigest(),\n  \'note\':\'Original PACE ZIP is NOT required to run: archived code SHA is documented. GA ablations and diversity selection are an extension of original workflow.\'}\n(OUT/\'full_protocol_model_provenance.json\').write_text(json.dumps(provenance,indent=2))\nALPHABET=list(\'ADEFGHIKLNPQRSTVWY\');LMIN=12;LMAX=30;POP=240;GENS=45;ELITE=24\nTARGETS=list(reg)\ndef fitness(v,arm):\n    a=2*v[\'p_amp\']-v[\'mic_mean\'];b=.5*v[\'mic_worst\'];c=.6*float(np.clip(v[\'ti\'],-2,4));d=2*max(0,v[\'sim\']-.55);e=.35*max(0,abs(v[\'charge\']-6)-3)\n    if arm==\'full\':return a-b+c-d-e\n    if arm==\'no_hemolysis\':return a-b-d-e\n    if arm==\'no_novelty\':return a-b+c-e\n    if arm==\'no_worst_case\':return a+c-d-e\n    if arm==\'potency_recognition_only\':return a-e\n    raise KeyError(arm)\n\ndef random_seq(rng):return \'\'.join(rng.choice(ALPHABET,int(rng.integers(LMIN,LMAX+1))))\ndef mutate(s,rng):\n    s=list(s);op=rng.random()\n    if op<.70 or len(s)<=LMIN:\n        for _ in range(int(rng.integers(1,4))):s[int(rng.integers(len(s)))]=str(rng.choice(ALPHABET))\n    elif op<.85 and len(s)<LMAX:s.insert(int(rng.integers(len(s)+1)),str(rng.choice(ALPHABET)))\n    else:del s[int(rng.integers(len(s)))]\n    return \'\'.join(s)\ndef crossover(a,b,rng):\n    s=a[:int(rng.integers(1,len(a)))]+b[int(rng.integers(1,len(b))):]\n    if len(s)>LMAX:s=s[:LMAX]\n    while len(s)<LMIN:s+=str(rng.choice(ALPHABET))\n    return s\n\ndef pair_sim(a,b):\n    u=set(a[i:i+3] for i in range(len(a)-2));v=set(b[i:i+3] for i in range(len(b)-2))\n    return len(u&v)/len(u|v)\n\ndef select_leads(unique,arm):\n    order=sorted(unique,key=lambda s:fitness(cache[s],arm),reverse=True)\n    passed=[];rejected_by=collections.Counter()\n    for s in order:\n        v=cache[s]\n        if v[\'p_amp\']<.9:rejected_by[\'p_amp\']+=1;continue\n        if v[\'mic_worst\']>np.log10(32):rejected_by[\'mic\']+=1;continue\n        if v[\'ti\']<1:rejected_by[\'selectivity\']+=1;continue\n        if v[\'sim\']>=.55:rejected_by[\'novelty\']+=1;continue\n        passed.append(s)\n    leads=[]\n    for s in passed:\n        if all(pair_sim(s,t)<=.45 for t in leads):leads.append(s)\n        if len(leads)==20:break\n    return leads,len(passed),dict(rejected_by)\n\ndef run(arm,seed):\n    rng=np.random.default_rng(seed)\n    pop=[random_seq(rng) for _ in range(POP)];seen=set();history=[]\n    for g in range(1,GENS+1):\n        rec=get_scores(pop);seen.update(pop)\n        fs=[fitness(s,arm) for s in rec]\n        order=np.argsort(-np.array(fs)); pop=[pop[j] for j in order];sc=[rec[j] for j in order];fs=[fs[j] for j in order]\n        history.append({\'arm\':arm,\'seed\':seed,\'gen\':g,\'best_native_fitness\':fs[0],\'population_mean_p_amp\':float(np.mean([s[\'p_amp\'] for s in sc])),\n          \'population_mean_mic_log10\':float(np.mean([s[\'mic_mean\'] for s in sc])),\'population_mean_ti_log10\':float(np.mean([s[\'ti\'] for s in sc])),\n          \'mean_length\':float(np.mean([s[\'length\'] for s in sc])),\'mean_novelty_similarity\':float(np.mean([s[\'sim\'] for s in sc]))})\n        if g==GENS:break\n        nxt=pop[:ELITE]\n        while len(nxt)<POP:\n            def tournament():\n                ids=rng.integers(0,len(pop),3)\n                return pop[int(max(ids,key=lambda i:fs[i]))]\n            c=crossover(tournament(),tournament(),rng) if rng.random()<.6 else tournament()\n            if rng.random()<.9:c=mutate(c,rng)\n            if LMIN<=len(c)<=LMAX:nxt.append(c)\n        pop=nxt\n    leads,feasible,rej=select_leads(seen,arm)\n    values=[cache[s] for s in leads]\n    out={\'arm\':arm,\'seed\':seed,\'unique_scored\':len(seen),\'feasible_hard_filters\':feasible,\'n_diverse_leads\':len(leads),\n         \'best_native_fitness\':max(fitness(cache[s],arm) for s in seen),\n         \'fraction_known_sim_above_0_55\':np.mean([cache[s][\'sim\']>.55 for s in seen]),\n         \'fraction_length_29_30_all\':np.mean([cache[s][\'length\']>=29 for s in seen]),\n         \'lead_avg_length\':float(np.mean([v[\'length\'] for v in values])) if values else None,\n         \'lead_avg_mic_log10\':float(np.mean([v[\'mic_mean\'] for v in values])) if values else None,\n         \'lead_avg_worst_mic_log10\':float(np.mean([v[\'mic_worst\'] for v in values])) if values else None,\n         \'lead_avg_ti_log10\':float(np.mean([v[\'ti\'] for v in values])) if values else None,\n         \'lead_avg_p_amp\':float(np.mean([v[\'p_amp\'] for v in values])) if values else None,\n         \'lead_avg_known_sim\':float(np.mean([v[\'sim\'] for v in values])) if values else None,\n         \'lead_avg_full_fitness\':float(np.mean([fitness(v,\'full\') for v in values])) if values else None,\n         \'filter_reject_reasons\':rej}\n    print(\'RUN\',json.dumps({k:v for k,v in out.items() if k!=\'filter_reject_reasons\'}),flush=True)\n    return out,history,[{\'arm\':arm,\'seed\':seed,\'rank\':i+1,**cache[s], \'full_fitness\':fitness(cache[s],\'full\'), \'native_fitness\':fitness(cache[s],arm)} for i,s in enumerate(leads)],seen\n\ndef save_table(path,rows):\n    if not rows:return\n    cols=list(rows[0]);\n    with path.open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,cols,extrasaction=\'ignore\');w.writeheader();w.writerows(rows)\n\n'
    exec(compile(SETUP_SOURCE,'PACE_Phase9_Refit_Predictors.py','exec'),globals())
print('Model suite ready: 4 classifier members,',len(reg),'MIC predictors, 1 HC50 predictor')


In [ ]:
# Load the controlled comparison functions into the active session.
RUNNER_SOURCE='"""PACE Phase 10B: controlled fixed-weight GA versus four-objective NSGA-II.\nThis module expects the Phase 9 refitted predictor functions and data (get_scores,\ncache, random_seq, mutate, crossover, pair_sim, fitness) in the same runtime.\nAll outputs are ANALYSIS-ONLY; user manuscript and supplementary files untouched.\n"""\nimport csv, json, time, hashlib, zipfile, math\nfrom pathlib import Path\nimport numpy as np\n\nOUT10B=Path(\'/content/pace_phase10b_colab\')\nOUT10B.mkdir(exist_ok=True,parents=True)\nSEEDS_10B=(7,29,41,53,67,79,97,113,131,149)\nUNIQUE_BUDGET=9000\nPOP10B=240\nGEN_MAX10B=45\nREF10B=np.array([np.log10(32),np.log10(32),-1.0],dtype=float)\nTOL10B=1e-11\n\ndef feasible10b(v):\n    return v[\'p_amp\']>=.9 and v[\'mic_worst\']<=REF10B[1] and v[\'ti\']>=1.0 and v[\'sim\']<.55\n\ndef obj10b(v):\n    # P(AMP) is a feasibility constraint, and excessive charge is soft penalized\n    # in scalar GA. NSGA-II trades predicted potency, selectivity and novelty.\n    return [v[\'mic_mean\'],v[\'mic_worst\'],-v[\'ti\'],v[\'sim\']]\n\ndef cv10b(v):\n    return max(0.,.9-v[\'p_amp\']) + max(0.,v[\'mic_worst\']-REF10B[1])+max(0.,1-v[\'ti\'])+max(0.,v[\'sim\']-.55)\n\ndef evaluate10b(pop,archive,budget=UNIQUE_BUDGET):\n    # Predicted sequences already in another algorithm\'s global cache STILL\n    # count against this run\'s independent unique-sequence budget.\n    fresh=list(dict.fromkeys(s for s in pop if s not in archive))\n    n=max(0,budget-len(archive));new=fresh[:n]\n    if new:\n        get_scores(new)\n        archive.update(new)\n    return [s for s in pop if s in archive],len(new)\n\ndef rank_crowd10b(pop):\n    # Small set (<=480); Deb-style constrained domination and standard crowding.\n    rec=get_scores(pop);X=np.asarray([obj10b(v) for v in rec],float)\n    cv=np.asarray([cv10b(v) for v in rec]);n=len(pop)\n    f=cv<=TOL10B\n    d=np.all(X[:,None,:]<=X[None,:,:]+TOL10B,axis=2)&np.any(X[:,None,:]<X[None,:,:]-TOL10B,axis=2)\n    dom=np.where(f[:,None]&~f[None,:],True,np.where(~f[:,None]&f[None,:],False,\n           np.where(~f[:,None]&~f[None,:],cv[:,None]<cv[None,:]-TOL10B,d)))\n    np.fill_diagonal(dom,False)\n    rank=np.full(n,-1,dtype=int);remain=np.ones(n,dtype=bool);fronts=[];idx=0\n    while remain.any():\n        ii=np.flatnonzero(remain)\n        counts=np.sum(dom[np.ix_(ii,ii)],axis=0)\n        front=ii[counts==0]\n        if not len(front):raise ValueError(\'Non-dominated ranking cycle\')\n        rank[front]=idx;remain[front]=False;fronts.append(front);idx+=1\n    crowd=np.zeros(n,dtype=float)\n    for front in fronts:\n        if len(front)<=2:crowd[front]=np.inf;continue\n        for j in range(X.shape[1]):\n            order=front[np.argsort(X[front,j],kind=\'stable\')]\n            spread=X[order[-1],j]-X[order[0],j]\n            if spread>TOL10B:\n                crowd[order[0]]=np.inf;crowd[order[-1]]=np.inf\n                crowd[order[1:-1]]+=(X[order[2:],j]-X[order[:-2],j])/spread\n    return rank,crowd,fronts\n\ndef choose_nsga10b(pop,k=POP10B):\n    pop=list(dict.fromkeys(pop));r,c,fronts=rank_crowd10b(pop);picked=[]\n    for front in fronts:\n        if len(picked)+len(front)<=k:picked.extend(front.tolist())\n        else:\n            order=front[np.argsort(-c[front],kind=\'stable\')]\n            picked.extend(order[:k-len(picked)].tolist());break\n    return [pop[i] for i in picked]\n\ndef tournament10b(pop,rank,crowd,rng):\n    i,j=rng.integers(0,len(pop),size=2)\n    if rank[i]!=rank[j]:return pop[i if rank[i]<rank[j] else j]\n    if crowd[i]!=crowd[j]:return pop[i if crowd[i]>crowd[j] else j]\n    return pop[int(i if rng.random()<.5 else j)]\n\ndef run_scalar10b(seed,budget=UNIQUE_BUDGET,popsize=POP10B,maxgens=GEN_MAX10B):\n    rng=np.random.default_rng(seed);pop=[random_seq(rng) for _ in range(popsize)];seen=set();hist=[]\n    for gen in range(1,maxgens+1):\n        scored,new=evaluate10b(pop,seen,budget)\n        if not scored:break\n        rec=get_scores(scored);ff=np.asarray([fitness(v,\'full\') for v in rec]);ix=np.argsort(-ff,kind=\'stable\')\n        ranked=[scored[i] for i in ix];ordered_f=ff[ix]\n        hist.append(dict(algorithm=\'Fixed-weight GA\',seed=seed,generation=gen,unique_evaluated=len(seen),\n            feasible_seen=sum(feasible10b(cache[s]) for s in seen),best_fitness=float(ordered_f[0]),\n            population_mean_mic=float(np.mean([v[\'mic_mean\'] for v in rec]))))\n        if len(seen)==budget or gen==maxgens:break\n        nxt=ranked[:min(24,len(ranked))]\n        while len(nxt)<popsize:\n            def tourn():\n                chosen=rng.integers(0,len(ranked),size=3)\n                return ranked[int(chosen[np.argmax(ordered_f[chosen])])]\n            s=crossover(tourn(),tourn(),rng) if rng.random()<.6 else tourn()\n            if rng.random()<.9:s=mutate(s,rng)\n            if 12<=len(s)<=30:nxt.append(s)\n        pop=nxt\n    return seen,hist\n\ndef run_nsga10b(seed,budget=UNIQUE_BUDGET,popsize=POP10B,maxgens=GEN_MAX10B):\n    rng=np.random.default_rng(seed);pop=[random_seq(rng) for _ in range(popsize)];seen=set();hist=[]\n    pop,_=evaluate10b(pop,seen,budget)\n    for gen in range(1,maxgens+1):\n        rank,crowd,fronts=rank_crowd10b(pop)\n        hist.append(dict(algorithm=\'NSGA-II\',seed=seed,generation=gen,unique_evaluated=len(seen),\n             feasible_seen=sum(feasible10b(cache[s]) for s in seen),first_front=len(fronts[0]),\n             feasible_population=sum(feasible10b(cache[s]) for s in pop)))\n        if len(seen)==budget or gen==maxgens:break\n        children=[]\n        while len(children)<popsize:\n            a=tournament10b(pop,rank,crowd,rng)\n            s=crossover(a,tournament10b(pop,rank,crowd,rng),rng) if rng.random()<.6 else a\n            if rng.random()<.9:s=mutate(s,rng)\n            if 12<=len(s)<=30:children.append(s)\n        children,_=evaluate10b(children,seen,budget)\n        if len(seen)==budget:break\n        pop=choose_nsga10b(pop+children,popsize)\n        while len(pop)<popsize:\n            s=random_seq(rng)\n            if s not in pop:\n                # Enforce budget when refreshing a collapsed population\n                s2,_=evaluate10b([s],seen,budget)\n                if not s2:break\n                pop.append(s)\n    return seen,hist\n\ndef pareto_front10b(strings,indices=(0,1,2,3)):\n    # Exact nondominated archive for these four objectives, without quadratic NxN allocation.\n    ss=list(strings)\n    if not ss:return [],np.empty((0,len(indices)))\n    X=np.asarray([obj10b(cache[s]) for s in ss])[:,indices]\n    sortkey=tuple(X[:,j] for j in range(len(indices)-1,-1,-1))\n    order=np.lexsort(sortkey);front=[]\n    for i in order:\n        if front:\n            # Lexicographic ascending ensures later items cannot dominate previous ones.\n            B=X[front]\n            if np.any(np.all(B<=X[i]+TOL10B,axis=1) & np.any(B<X[i]-TOL10B,axis=1)):\n                continue\n            if np.any(np.all(abs(B-X[i])<=TOL10B,axis=1)):\n                continue\n        front.append(i)\n    return [ss[i] for i in front],X[front]\n\ndef crowd_front10b(X):\n    n,m=X.shape\n    c=np.zeros(n,dtype=float)\n    if n<=2:return np.full(n,np.inf)\n    for j in range(m):\n        order=np.argsort(X[:,j],kind=\'stable\');spread=X[order[-1],j]-X[order[0],j]\n        if spread>1e-12:\n            c[order[0]]=np.inf;c[order[-1]]=np.inf\n            c[order[1:-1]]+=(X[order[2:],j]-X[order[:-2],j])/spread\n    return c\n\ndef common_leads10b(seen,policy):\n    feasible=[s for s in seen if feasible10b(cache[s])]\n    feasible.sort()\n    fallback=0\n    if policy==\'scalar\':order=sorted(feasible,key=lambda s:(-fitness(cache[s],\'full\'),s))\n    elif policy==\'pareto\':\n        front,X=pareto_front10b(feasible)\n        crowd=crowd_front10b(X)\n        order=[front[i] for i in sorted(range(len(front)),key=lambda i:(-crowd[i],-fitness(cache[front[i]],\'full\'),front[i]))]\n        remain=sorted(set(feasible)-set(front),key=lambda s:(-fitness(cache[s],\'full\'),s))\n        order+=remain\n    else:raise ValueError(policy)\n    frontset=set(order[:len(front)]) if policy==\'pareto\' else set()\n    leads=[]\n    for s in order:\n        if all(pair_sim(s,t)<=.45 for t in leads):\n            leads.append(s)\n            if policy==\'pareto\' and s not in frontset:fallback+=1\n        if len(leads)>=20:break\n    return leads, {\'n_feasible\':len(feasible),\'n_selected\':len(leads),\'pareto_scalar_fallbacks\':fallback}\n\ndef hv2_10b(P,ref):\n    if not len(P):return 0.\n    P=P[np.all(P<ref,axis=1)]\n    if not len(P):return 0.\n    P=P[np.argsort(P[:,0])];best=ref[1];volume=0.\n    for i,row in enumerate(P):\n        best=min(best,row[1]);nxt=P[i+1,0] if i+1<len(P) else ref[0]\n        volume+=max(0.,nxt-row[0])*max(0.,ref[1]-best)\n    return float(volume)\n\ndef hv3_10b(seen):\n    feas=[s for s in seen if feasible10b(cache[s])]\n    if not feas:return 0.,0\n    front,X=pareto_front10b(feas,indices=(0,1,2))\n    X=X[np.all(X<REF10B,axis=1)]\n    ux=np.unique(X[:,0]);vol=0.\n    for i,x in enumerate(ux):\n        nxt=ux[i+1] if i+1<len(ux) else REF10B[0]\n        vol+=(nxt-x)*hv2_10b(X[X[:,0]<=x][:,1:],REF10B[1:])\n    return float(vol),len(front)\n\ndef metrics10b(rows):\n    return {\n     \'mean_predicted_mic_um\':float(np.mean([10**v[\'mic_mean\'] for v in rows])),\n     \'mean_predicted_worst_mic_um\':float(np.mean([10**v[\'mic_worst\'] for v in rows])),\n     \'mean_predicted_selectivity\':float(np.mean([10**v[\'ti\'] for v in rows])),\n     \'mean_amp_probability\':float(np.mean([v[\'p_amp\'] for v in rows])),\n     \'mean_known_3mer_similarity\':float(np.mean([v[\'sim\'] for v in rows])),\n     \'median_pairwise_3mer_similarity\':float(np.median([pair_sim(a[\'sequence\'],b[\'sequence\']) for i,a in enumerate(rows) for b in rows[i+1:]])),\n     \'mean_length\':float(np.mean([v[\'length\'] for v in rows]))\n    }\n\ndef write_csv10b(p,rows):\n    if not rows:return\n    keys=list(rows[0].keys())\n    with open(p,\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=keys,extrasaction=\'ignore\');w.writeheader();w.writerows(rows)\n\ndef run_one10b(seed):\n    output=[];histories=[]\n    for name,fn in [(\'Fixed-weight GA\',run_scalar10b),(\'NSGA-II\',run_nsga10b)]:\n        t=time.time();archive,hist=fn(seed);histories+=hist\n        if len(archive)!=UNIQUE_BUDGET:raise RuntimeError(f\'{name} seed {seed}: only {len(archive)} unique evaluated, required {UNIQUE_BUDGET}; increase MAXGENS.\')\n        hv,nd=hv3_10b(archive)\n        feas=[s for s in archive if feasible10b(cache[s])]\n        front,_=pareto_front10b(feas)\n        for policy in (\'scalar\',\'pareto\'):\n            selected,meta=common_leads10b(archive,policy)\n            if len(selected)<20:raise RuntimeError(f\'{name}/{seed}/{policy} only {len(selected)} feasible diverse leads\')\n            rec=[cache[s] for s in selected]\n            vals=metrics10b(rec)\n            output.append(dict(algorithm=name,seed=seed,selection_policy=policy,unique_evaluated=len(archive),\n                     n_feasible=len(feas),n_4d_nondominated_archive=len(front),n_3d_nondominated_archive=nd,\n                     hypervolume_3d=hv,pareto_scalar_fallbacks=meta[\'pareto_scalar_fallbacks\'],\n                     runtime_seconds=time.time()-t,**vals))\n            leadrows=[dict(algorithm=name,seed=seed,selection_policy=policy,rank=i+1,\n                    **v,full_fitness=fitness(v,\'full\'),predicted_geom_mean_MIC_um=10**v[\'mic_mean\'],\n                    predicted_worst_MIC_um=10**v[\'mic_worst\'],predicted_selectivity=10**v[\'ti\'])\n                 for i,v in enumerate(rec)]\n            write_csv10b(OUT10B/f\'{name.replace(" ","_")}_{policy}_{seed}_leads.csv\',leadrows)\n        # Save an entire archive for an audit by third party\n        write_csv10b(OUT10B/f\'{name.replace(" ","_")}_{seed}_full_archive.csv\',\n                    [cache[s] for s in sorted(archive)])\n        print(\'Phase10B\',name,\'seed\',seed,\'N\',len(archive),\'hv3\',round(hv,5),\'elapsed\',round(time.time()-t,1),\'s\',flush=True)\n    write_csv10b(OUT10B/f\'seed_{seed}_summary.csv\',output)\n    write_csv10b(OUT10B/f\'seed_{seed}_history.csv\',histories)\n    return output\n\ndef aggregate10b():\n    import pandas as pd\n    paths=sorted(OUT10B.glob(\'seed_*_summary.csv\'))\n    if not paths:raise RuntimeError(\'No seed summaries\')\n    df=pd.concat([pd.read_csv(p) for p in paths],ignore_index=True)\n    assert len(df)==4*len(paths),f\'Missing results: {len(df)} records for {len(paths)} seeds\'\n    df.to_csv(OUT10B/\'all_seed_summary.csv\',index=False)\n    metric_cols=[\'hypervolume_3d\',\'mean_predicted_mic_um\',\'mean_predicted_worst_mic_um\',\'mean_predicted_selectivity\',\n                 \'mean_amp_probability\',\'mean_known_3mer_similarity\',\'median_pairwise_3mer_similarity\']\n    rng=np.random.default_rng(2026)\n    stats=[]\n    for policy in (\'scalar\',\'pareto\'):\n        subset=df[df.selection_policy==policy]\n        for met in metric_cols:\n            p=subset.pivot(index=\'seed\',columns=\'algorithm\',values=met)\n            delta=(p[\'NSGA-II\']-p[\'Fixed-weight GA\']).to_numpy()\n            boot=delta[rng.integers(0,len(delta),(10000,len(delta)))].mean(axis=1)\n            stats.append(dict(selection_policy=policy,metric=met,n_pairs=len(delta),\n                mean_nsga_minus_fixed=float(delta.mean()),median_nsga_minus_fixed=float(np.median(delta)),\n                bootstrap95_low=float(np.quantile(boot,.025)),bootstrap95_high=float(np.quantile(boot,.975)),\n                note=\'Paired seeds; exploratory uncertainty, not experimental validation.\'))\n    pd.DataFrame(stats).to_csv(OUT10B/\'paired_seed_bootstrap.csv\',index=False)\n    meta={\'source_phase8_sha256\':hashlib.sha256(Path(\'/content/PACE_Phase8_ESM2_results.zip\').read_bytes()).hexdigest(),\n          \'trained_model_source\':\'Full Phase 9 refitted PACE 90D models, same 7 prediction models for both optimizers\',\n          \'seeds\':list(SEEDS_10B),\'unique_evaluation_budget_per_algorithm_per_seed\':UNIQUE_BUDGET,\n          \'population\':POP10B,\'maximum_generations\':GEN_MAX10B,\n          \'initial_population\':\'Matched via same seed; same random_seq function, objective-independent\',\n          \'archive_selection\':\'Both algorithms: all evaluated unique sequences and the identical feasibility/diversity rules\',\n          \'selection_policies\':[\'scalar ranking by original full PACE fitness\',\'4D Pareto crowding then scalar fallback\'],\n          \'Pareto_4D\':[\'mean log10MIC\',\'worst log10MIC\',\'negative log10 selectivity\',\'3mer known-corpus similarity\'],\n          \'NSGA2_constraint_rules\':\'P(AMP)>=0.9; worst MIC<=32uM; TI>=10; known similarity<0.55\',\n          \'hypervolume_3D_reference\':REF10B.tolist(),\n          \'comparison_scope\':\'In silico; MIC, hemolysis and selectivity are all predictions, no biological validation\',\n          \'important\':\'Different numbers of generations are allowed to achieve exactly the same 9000 unique-evaluation budget.\'}\n    (OUT10B/\'protocol_metadata.json\').write_text(json.dumps(meta,indent=2))\n    out=Path(\'/content/PACE_Phase10B_Controlled_Results.zip\')\n    with zipfile.ZipFile(out,\'w\',zipfile.ZIP_DEFLATED,compresslevel=6) as z:\n        for p in OUT10B.glob(\'*\'):\n            if p.is_file():z.write(p,arcname=p.name)\n    print(\'Phase10B results ready:\',out,\'bytes\',out.stat().st_size,flush=True)\n    return out\n\n'
exec(compile(RUNNER_SOURCE,'PACE_Phase10B_Controlled_Runner.py','exec'),globals())
print('10 paired seeds:',SEEDS_10B)
print('Unique-sequence budget per algorithm per seed:',UNIQUE_BUDGET)
print('4D Pareto objectives:', ['mean_log10_MIC','worst_log10_MIC','negative_log10_selectivity','known_3mer_similarity'])


In [ ]:
# True full-budget evolutionary experiments. Each completed seed is checkpointed.
from pathlib import Path
for seed in SEEDS_10B:
    result=OUT10B/f'seed_{seed}_summary.csv'
    if result.exists():
        print('Verified checkpoint exists; skipping seed',seed,flush=True)
        continue
    print('Starting controlled paired seed',seed,flush=True)
    run_one10b(seed)
print('All requested paired seeds completed')


In [ ]:
# Statistical summary, full audit archives, protocol, final ZIP and download.
from google.colab import files
out=aggregate10b()
files.download(str(out))


### Verification notes

- If there is an interruption, **do not restart the Colab runtime**; rerun only the experimental cell, which skips seeds already saved.
- Keep the `PACE_Phase10B_Controlled_Results.zip` file intact and upload it into our conversation.
- Both methods are compared using the same retrained PACE models; unlike the earlier two-seed comparison, this run matches the **unique prediction evaluation count** and uses **two common selection policies**.
- The full archive files are intentionally included for independent numerical verification of the Pareto statistics.
- The hypervolume calculation uses **three quantitative objectives** (mean MIC, worst MIC, predicted selectivity) so that it does not become dominated by a nearly constant charge/recognition coordinate. The NSGA optimizer itself uses four objectives, including known-sequence novelty.
